In [1]:
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('../')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully
Table VERSION_TRACKER created successfully
Table METRICS_TRACKER created successfully

Share anonymous install statistics? (opt-out instructions)

PixieDust will record metadata on its environment the next time the package is installed or updated. The data is anonymized and aggregated to help plan for future releases, and records only the following values:

{
   "data_sent": currentDate,
   "runtime": "python",
   "application_version": currentPixiedustVersion,
   "space_id": nonIdentifyingUniqueId,
   "config": {
       "repository_id": "https://github.com/ibm-watson-data-lab/pixiedust",
       "target_runtimes": ["Data Science Experience"],
       "event_id": "web",
       "event_organizer": "dev-journeys"
   }
}
You can opt out by calling pixiedust.optOut() in a new cell.


Pixiedust runtime updated. Please restart kernel
Table SPARK_PACKAGES created successfully
Table USER_PREFERENCES created successfully
Table service_connections created successfully
Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
     |████████████████████████████████| 15.9 MB 26.1 MB/s eta 0:00:01
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [3]:
!pwd

/home/o_suchsi/work/Oklahoma State/Priya/epilepsy


In [4]:
medication_s = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Zheng_Han/medication_s.parquet')

In [5]:
medication_s.count()

2996074

In [9]:
medication_s.show(5)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------------------+--------------------+--------------------+--------------------+--------------------+---------------------+--------------------+--------------------+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------+--------+------+
|        medicationid|         encounterid|            personid|   intendeddispenser|           startdate|intendedadministrator|            doseunit|            stopdate|            category|active|           frequency|              status|               route|            drugcode| prescribingprovider|dosequantity|source|asneeded|tenant|
+--------------------+--------------------+--------------------+--------------------+--------------------+---------------------+--------------------+--------------------+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+----

<IPython.core.display.Javascript object>

In [10]:
medication_s.printSchema()

▸,:,


root
 |-- medicationid: string (nullable = true)
 |-- encounterid: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- intendeddispenser: string (nullable = true)
 |-- startdate: string (nullable = true)
 |-- intendedadministrator: string (nullable = true)
 |-- doseunit: struct (nullable = true)
 |    |-- standard: struct (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- codingSystemId: string (nullable = true)
 |    |    |-- primaryDisplay: string (nullable = true)
 |    |-- standardCodings: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- codingSystemId: string (nullable = true)
 |    |    |    |-- primaryDisplay: string (nullable = true)
 |-- stopdate: string (nullable = true)
 |-- category: struct (nullable = true)
 |    |-- standard: struct (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- codingSystemId: string (n

In [3]:
medication_s.createOrReplaceTempView('medication_s')


▸,:,


In [4]:
ndclist = ['137405680','137424045','137433573','062623845','062623855',\
           '062623870','062623905','137493221','137494114','137497136',\
          '137402037','137401216','137401319','137498131','137401275']

▸,:,


In [11]:
druglikelist = ['carbamazepine', 'phenytoin','valproic','phenobarbital','felbamate','gabapentin','lamotrigine','levetiracetam',\
               'oxcarbazepine','tiagabine','topiramate','zonisamide']


▸,:,


In [22]:
def extract_pt_from_med(spark, druglikelist=[],outputfilename="", outputfolder = ""):
#     items in icdlist and icdlikes should be single quote mark
# the outputfolder should be like "Zheng_Han/pompe". It is appended within the Oklahoma state folder. No relative path is allowed.

    if outputfilename =="":
        print("please set outputfilename!")
        return None
    
    if outputfolder =="":
        print("please set outputfolder!")
        return None
    
            
    sql_sentence = "select distinct encounterid, personid, drugcode.standard.id as drugid, drugcode.standard.primaryDisplay as drugname, startdate, stopdate from medication where "

    likesentence = ""
    
    
    
    if len(druglikelist) > 0:
        likesentence = "drugcode.standard.primaryDisplay like "
        for i in range(len(druglikelist)-1):
            likesentence = likesentence + "'%" + druglikelist[i] + "%'" 
            likesentence += " or drugcode.standard.primaryDisplay like "

        likesentence = likesentence + "'%" + druglikelist[-1] +  "%'" 
        
   
    sql_sentence = sql_sentence + likesentence
          
    print(sql_sentence)
    df = spark.sql(sql_sentence)
    df.write.mode('overwrite').parquet(outputfilename)
    df.cache()
    os.system("hadoop fs -copyToLocal " + outputfilename + " ~/work/Oklahoma%20State/" + outputfolder)
    return df

▸,:,


In [6]:
selected_med = spark.sql("""
select encounterid, personid, drugcode.standard.id as drugid, drugcode.standard.primaryDisplay as drugname, startdate, stopdate
from medication_s
where drugcode.standard.primaryDisplay like '%arbamazepine%'
""")

▸,:,


In [7]:
selected_med.show()

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------------------+--------------------+--------------+--------------------+--------------------+--------------------+
|         encounterid|            personid|        drugid|            drugname|           startdate|            stopdate|
+--------------------+--------------------+--------------+--------------------+--------------------+--------------------+
|30f2f25e-13d4-49d...|d433d0ee-6249-452...|          2002|       Carbamazepine| 2018-04-24T00:00:00|                    |
|eeedc160-1056-4fc...|8576dbca-f99f-44b...|          2002|       Carbamazepine| 2017-03-14T00:00:00|                    |
|7e3a198e-ac1d-452...|2a0b5c52-8487-452...|          2002|       carbamazepine| 2015-08-25T00:00:00|                    |
|5891c803-9dd0-4cf...|8eb836d9-8603-4a8...|          2002|       carbamazepine| 2015-06-09T00:00:00|                    |
|19c95f97-325c-40a...|b3c9bef3-d76f-49f...|          2002|       carbamazepine| 2018-11-21T00:00:00|                    |
|209d95af-48d3-452...|3c

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
epi_med = extract_pt_from_med(spark, druglikelist=druglikelist,outputfilename="epi_med", outputfolder = "Zheng_Han/epilepsy")

▸,:,


select encounterid, personid, drugcode.standard.id as drugid, drugcode.standard.primaryDisplay as drugname, startdate, stopdate from medication where drugcode.standard.primaryDisplay like '%carbamazepine%' or drugcode.standard.primaryDisplay like '%phenytoin%' or drugcode.standard.primaryDisplay like '%valproic%' or drugcode.standard.primaryDisplay like '%phenobarbital%' or drugcode.standard.primaryDisplay like '%felbamate%' or drugcode.standard.primaryDisplay like '%gabapentin%' or drugcode.standard.primaryDisplay like '%lamotrigine%' or drugcode.standard.primaryDisplay like '%levetiracetam%' or drugcode.standard.primaryDisplay like '%oxcarbazepine%' or drugcode.standard.primaryDisplay like '%tiagabine%' or drugcode.standard.primaryDisplay like '%topiramate%' or drugcode.standard.primaryDisplay like '%zonisamide%'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
epi_med.cache()
epi_med.show(5)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------------------+--------------------+------+----------+--------------------+--------------------+
|         encounterid|            personid|drugid|  drugname|           startdate|            stopdate|
+--------------------+--------------------+------+----------+--------------------+--------------------+
|14b11500-d1b9-498...|6c1e3bb2-a0fd-4ef...|d03182|gabapentin|2006-01-10T05:00:...|2006-01-20T23:01:...|
|1cfa4e98-20f8-410...|1434fd1e-ddfd-426...|d00143| phenytoin|2008-01-20T17:00:...|2008-01-23T04:02:...|
|1cfa4e98-20f8-410...|1434fd1e-ddfd-426...|d00143| phenytoin|2008-01-20T04:37:...|2008-01-20T09:40:...|
|f27b6449-f7fe-47d...|05155310-4483-4be...|d00143| phenytoin|2008-12-05T05:00:...|2008-12-05T21:00:...|
|f27b6449-f7fe-47d...|05155310-4483-4be...|d00143| phenytoin|2008-12-04T06:00:...|2008-12-05T01:14:...|
+--------------------+--------------------+------+----------+--------------------+--------------------+
only showing top 5 rows



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
epi_med = spark.read.parquet('file:/home/z_han/work/Oklahoma State/Zheng_Han/epilepsy/epi_med')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
etl.cohort_statistics(spark, epi_med)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The number of unique encounters:  11996355


<IPython.core.display.Javascript object>